# Week 4: Build a Tiny LLM From Scratch

## Learning Objectives

By the end of this session you will be able to:

- Take the positional encoding formula, the self-attention mechanism, and the `MultiHeadAttention` class from Week 3 and use them, unchanged, as working components of a real model
- Implement causal masking, the one piece of math Week 3 deliberately left out, and prove numerically that it blocks a token from seeing the future
- Assemble a full transformer block (attention plus residual connections, layer norm, and a feedforward network) and stack several into `TinyTransformerLM`
- Explain the language modeling objective (next-token prediction) as cross-entropy loss, and implement the shifted input/target pairs that make it trainable
- Train `TinyTransformerLM` from randomly initialized weights on real text, and watch loss fall and generated text improve
- Explain honestly what separates this toy model from GPT-2 or GPT-3, and why that gap is compute and data, not a missing idea


## 1. Bringing From Week 3

Three pieces come forward unchanged. Each one below is exactly what you built last week, not a rewrite.


### 1.1 Positional encoding

This is the sine/cosine function from Week 3, Section 3, the one we hand-verified against $PE(1,:) \approx [0.8415, 0.5403, 0.0100, 0.99995]$.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)
np.random.seed(0)

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model // 2)
    div_term = 10000 ** (2 * i / d_model)
    pe[:, 0::2] = np.sin(position / div_term)
    pe[:, 1::2] = np.cos(position / div_term)
    return pe

# same check as Week 3
pe_check = positional_encoding(seq_len=3, d_model=4)
print("PE(1, :) =", pe_check[1].round(4), " matches Week 3's hand calc: [0.8415, 0.5403, 0.0100, 0.99995]")


PE(1, :) = [0.8415 0.5403 0.01   1.    ]  matches Week 3's hand calc: [0.8415, 0.5403, 0.0100, 0.99995]


### 1.2 Greedy and temperature-sampled decoding

These are the two decoding functions from Week 3, Section 2, unchanged. We'll use them directly inside the generation loop later, rather than writing new sampling logic.


In [2]:
def softmax_np(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def greedy_decode(logits, vocab):
    return vocab[np.argmax(logits)]

def sample_decode(logits, vocab, temperature=1.0, rng=None):
    rng = rng or np.random.default_rng()
    probs = softmax_np(logits / temperature)
    return rng.choice(vocab, p=probs), probs

# same sanity check as Week 3
vocab_toy = ["cat", "dog", "ran", "slept", "quickly"]
logits_toy = np.array([2.5, 2.3, 1.0, 0.5, -0.2])
print("greedy pick:", greedy_decode(logits_toy, vocab_toy))


greedy pick: cat


### 1.3 The `MultiHeadAttention` class

This is the exact class from Week 3, Section 5.2, including the `causal_mask` parameter in `forward`. Last week that parameter was always left as `None`, since we hadn't built a mask to pass in yet. That changes in Section 3 below.


In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must divide evenly into n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal_mask=None):
        batch, seq_len, d_model = x.shape

        Q = self.W_Q(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        head_outputs = weights @ V

        concat = head_outputs.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
        return self.W_O(concat), weights

# same shape check as Week 3
d_model, n_heads, seq_len, batch = 8, 2, 3, 1
mha_check = MultiHeadAttention(d_model, n_heads)
out, attn_weights = mha_check(torch.randn(batch, seq_len, d_model))
print("output shape:", out.shape, " attention weight shape:", attn_weights.shape)


output shape: torch.Size([1, 3, 8])  attention weight shape: torch.Size([1, 2, 3, 3])


## 2. Causal Masking

Week 3 flagged this and stopped: "it" was only ever compared against tokens that already existed at or before its own position, so the hand-calc examples were valid without an explicit mask. But a real model generates left to right, one new token at a time, and during training it processes an entire sequence in parallel. Without a mask, position 3 could peek at position 4's key and value while learning to predict position 4, which means the loss signal wouldn't reflect genuine prediction at all, it would reflect an open-book test.

A causal mask is simple: a lower-triangular matrix of 1s and 0s. Row $t$ has 1s in columns $0$ through $t$ (positions the token can see) and 0s afterward (positions it can't). `MultiHeadAttention.forward` already accepts this mask, we just never built one to pass in.


In [4]:
seq_len = 4
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("causal mask:\n", causal_mask)


causal mask:
 tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


Let's prove it actually changes something. Run the same random input through `MultiHeadAttention` with and without the mask, and look at the attention weights assigned by position 0 (the very first token) to every position in the sequence.


In [5]:
torch.manual_seed(0)
x_demo = torch.randn(1, seq_len, d_model)

_, weights_unmasked = mha_check(x_demo)
_, weights_masked = mha_check(x_demo, causal_mask=causal_mask)

print("position 0's attention weights, no mask:  ", weights_unmasked[0, 0, 0].detach().numpy().round(4))
print("position 0's attention weights, with mask:", weights_masked[0, 0, 0].detach().numpy().round(4))


position 0's attention weights, no mask:   [0.155  0.2795 0.38   0.1855]
position 0's attention weights, with mask: [1. 0. 0. 0.]


Without the mask, position 0 spreads attention across all four positions, including three that haven't "happened" yet from its point of view. With the mask, position 0's weight collapses entirely onto position 0, the only token it's allowed to see. This is the exact mechanism, applied to every row of the score matrix at once: row $t$ gets masked to only ever attend to columns $0 \ldots t$.


## 3. Assembling the Full Transformer Block

Attention alone isn't a transformer block. The original architecture wraps it with two more ingredients:

- **Residual connections**: `x = x + attn(x)` instead of `x = attn(x)`. This gives gradients a direct path backward through the network (helpful once we stack many blocks), and means each block only has to learn a *correction* to its input rather than reconstruct the whole representation from nothing.
- **Layer normalization**: rescales each token's vector to have stable mean and variance before it enters attention or the feedforward step. This keeps training numerically stable as we stack blocks deeper.

Alongside attention, each block also has a small position-wise feedforward network, a two-layer MLP applied identically to every position, giving the model extra capacity to transform each token's representation after attention has mixed in context from its neighbors.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, causal_mask):
        attn_out, _ = self.attn(self.ln1(x), causal_mask=causal_mask)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

# shape check: a block's output must match its input, so blocks can stack
block_check = TransformerBlock(d_model=8, n_heads=2)
mask_check = torch.tril(torch.ones(3, 3))
out = block_check(torch.randn(1, 3, 8), mask_check)
print("block output shape:", out.shape, "(matches input shape, as required for stacking)")


block output shape: torch.Size([1, 3, 8]) (matches input shape, as required for stacking)


## 4. `TinyTransformerLM`: Putting It All Together

Now every piece exists: token embeddings, Week 3's positional encoding, Week 3's `MultiHeadAttention` wrapped in a causally-masked `TransformerBlock`, and a final linear "LM head" that projects back to vocabulary-sized logits. `TinyTransformerLM` stacks several blocks and wires all of it together.

Notice the positional encoding is added once, right after the token embedding, using Week 3's function directly (not a new learned embedding, the same fixed sine/cosine pattern from last week). The causal mask is built once, sized to the longest sequence the model will ever see (`block_size`), and sliced down to whatever length the current input actually is.


In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=3, block_size=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model) # feed a token index and get back embedding vector

        pe = positional_encoding(block_size, d_model)  # Week 3's function
        self.register_buffer("pos_emb", torch.tensor(pe, dtype=torch.float32))

        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.block_size = block_size

        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb[:T].unsqueeze(0)
        x = tok + pos

        mask = self.causal_mask[:T, :T]
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

# shape and gradient smoke test before we touch real data
toy_model = TinyTransformerLM(vocab_size=59, d_model=64, n_heads=4, n_layers=3, block_size=32)
xb_check = torch.randint(0, 59, (16, 32))
yb_check = torch.randint(0, 59, (16, 32))
logits, loss = toy_model(xb_check, yb_check)
print("logits shape:", logits.shape, " loss:", round(loss.item(), 4))
print("total trainable parameters:", sum(p.numel() for p in toy_model.parameters()))


logits shape: torch.Size([16, 32, 59])  loss: 4.2397
total trainable parameters: 156923


One useful check before we ever train on real data: does gradient descent actually work through this whole stack? Fit the model to a single fixed random batch for a few dozen steps. It should overfit fast, since it's a handful of numbers with no real structure to generalize, but a shrinking loss confirms every piece (masking, attention, residuals, layer norm, the loss itself) is wired correctly and gradients are flowing.


In [8]:
optimizer_check = torch.optim.AdamW(toy_model.parameters(), lr=3e-3)
for step in range(50):
    logits, loss = toy_model(xb_check, yb_check)
    optimizer_check.zero_grad(set_to_none=True)
    loss.backward()
    optimizer_check.step()

print("loss after 50 steps memorizing one fixed random batch:", round(loss.item(), 4))
print("(started around 4.1, a large drop confirms gradients flow correctly through the whole stack)")


loss after 50 steps memorizing one fixed random batch: 0.0228
(started around 4.1, a large drop confirms gradients flow correctly through the whole stack)


## 5. The Language Modeling Objective

Everything above defines an architecture. It says nothing yet about what "correct" means. That's the objective: next-token prediction. Given every token up to position $t$, predict the token at position $t+1$. The text labels itself, there's no separate annotation step, the next character in real text is always sitting right there as the target.

In Week 1 we used binary cross-entropy (BCE) for a single yes/no output. Here the output at each position is a probability distribution over the whole vocabulary, so we need the multi-class generalization of the same idea: **cross-entropy loss**.

$$
\mathcal{L} = -\log P(y_{\text{true}})
$$

where $P(y_{\text{true}})$ is the probability the model assigned to the correct next token, after passing the logits through softmax. Same shape as BCE: confident and correct gives a small loss, confident and wrong gives a large one. BCE was the two-outcome special case of this all along.


### 5.1 Hand calculation: loss at a single position

Reusing Week 2's cat and dog theme, a tiny 4-word vocabulary:

| index | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| token | the | cat | dog | sleeps |

Suppose the model has just seen "the cat" and needs to predict the next token. The true next token is "sleeps" (index 3). Say the model's logits at this position are:

$$
z = [0.2,\ 0.1,\ 0.3,\ 0.4]
$$

**Step 1: softmax.**

$$
e^{0.2}=1.2214,\quad e^{0.1}=1.1052,\quad e^{0.3}=1.3499,\quad e^{0.4}=1.4918, \qquad \text{sum} = 5.1683
$$

$$
P = [0.2363,\ 0.2138,\ 0.2612,\ 0.2887]
$$

**Step 2: cross-entropy.** Only the true token's probability matters directly:

$$
\mathcal{L} = -\log(0.2887) = 1.2425
$$

A real training step doesn't stop at one position, it computes this same loss at every position in a sequence and averages them, the same way any mini-batch loss gets averaged (Week 1). That averaged number is what `F.cross_entropy` reports by default, and it's what you'll watch drop during training below.

One more useful conversion: **perplexity**, defined as $e^{\mathcal{L}}$. It has a clean intuition: a perplexity of $p$ means the model is about as confused as if it were guessing uniformly among $p$ options.


In [9]:
logits = torch.tensor([0.2, 0.1, 0.3, 0.4])
true_idx = 3  # "sleeps"

probs = F.softmax(logits, dim=0)
print("probs:", [round(p.item(), 4) for p in probs])

loss = -torch.log(probs[true_idx])
print("manual loss:", round(loss.item(), 4))

torch_loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([true_idx]))
print("F.cross_entropy loss:", round(torch_loss.item(), 4))
print("perplexity:", round(math.exp(torch_loss.item()), 4))


probs: [0.2363, 0.2138, 0.2612, 0.2887]
manual loss: 1.2425
F.cross_entropy loss: 1.2425
perplexity: 3.4644


## 6. Why the Shift Matters (a trap worth falling into on purpose)

One detail is easy to skim past: to train on "the cat sleeps", the input is `[the, cat]` and the target is `[cat, sleeps]`, the same sequence offset by one token. What happens if we get lazy and set target = input instead?

Because of the causal mask we just built, position $t$ can already see token $t$, it's part of its own input. If the target at position $t$ is also token $t$, the model doesn't need to understand language at all, it just needs to learn "copy whatever you see at your own position." Watch what happens to the loss when we hand-construct a "model" that does nothing but copy:


In [10]:
vocab_size_toy = 4  # 0=the, 1=cat, 2=dog, 3=sleeps
input_token = 1   # "cat"
target_token = 1  # same as input, because we forgot to shift

cheat_logits = torch.full((vocab_size_toy,), -100.0)
cheat_logits[input_token] = 100.0

loss = F.cross_entropy(cheat_logits.unsqueeze(0), torch.tensor([target_token]))
print("loss when target = input (no shift):", loss.item())


loss when target = input (no shift): 0.0


A loss of exactly zero looks fantastic on a training curve and means nothing, the model learned to echo, not to predict. This is why every language model training pipeline, from this notebook to GPT-4, shifts the target sequence by exactly one position relative to the input. It's the entire task definition: predict what you haven't seen yet, not what you have.


## 7. Real Data: Preparing a Text Corpus

Time to move off toy vocabularies and toy batches onto real text: the Tiny Shakespeare corpus, a standard small dataset for exactly this kind of from-scratch demo.

For tokenization we're going to the opposite extreme of Week 2's BPE tokenizer: **character-level** tokenization. Every unique character becomes one token. This keeps the vocabulary tiny (under 100 tokens) and the model small enough to train in a couple of minutes, while still learning genuine structure, spelling, punctuation, even the script's `CHARACTER NAME:` formatting. It sits at one end of the tokenization spectrum from Week 2, BPE sits in the middle, word-level tokenization at the other end.


In [12]:
with open("tinyshakespeare.txt") as f:
    text = f.read()

text = text[:50000]  # a slice is enough to see real learning happen quickly

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
vocab_list = [itos[i] for i in range(vocab_size)]  # ordered list, index i <-> character

def encode(s):
    return [stoi[c] for c in s]

def decode(idxs):
    return "".join(itos[i] for i in idxs)

print("vocab size:", vocab_size)
print("vocabulary:", "".join(chars))
print("encode('cat'):", encode("cat"))
print("decode back:", decode(encode("cat")))


vocab size: 59
vocabulary: 
 !',-.:;?ABCDEFGHIJKLMNOPRSTUVWYabcdefghijklmnopqrstuvwxyz
encode('cat'): [35, 33, 52]
decode back: cat


### 7.1 Building input/target batches

We hold out a small validation split, then sample random windows of length `block_size` from the training text. Exactly as Section 6 established: `x` is the window, `y` is the same window shifted one character to the right. `block_size` here is the context length, the same role it played in Week 3's positional encoding discussion. `batch_size` is how many independent windows we process at once (Week 1's mini-batch idea, applied to sequences).


In [13]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train tokens:", len(train_data), " val tokens:", len(val_data))

block_size = 32
batch_size = 16

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print("x shape:", xb.shape, " y shape:", yb.shape)
print("\nfirst training example, input: ", repr(decode(xb[0].tolist())))
print("first training example, target:", repr(decode(yb[0].tolist())))


train tokens: 45000  val tokens: 5000
x shape: torch.Size([16, 32])  y shape: torch.Size([16, 32])

first training example, input:  "aty find\nI' the part that is at "
first training example, target: "ty find\nI' the part that is at m"


## 8. The Training Loop

The same four-step loop from Week 1: forward pass, compute loss, backward pass, optimizer step. Nothing about training a transformer changes that mechanic, only the shapes involved and the objective.

We use `AdamW`, a variant of Adam that generally trains transformers more reliably than plain SGD, adapting the learning rate per-parameter based on the recent history of gradients. `estimate_loss` averages loss over several batches, both train and validation, since a single batch's loss is noisy and we want a clean trend to watch.


In [14]:
model = TinyTransformerLM(vocab_size, d_model=64, n_heads=4, n_layers=3, block_size=block_size)
n_params = sum(p.numel() for p in model.parameters())
print("total trainable parameters:", n_params)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

@torch.no_grad()
def estimate_loss(n_batches=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(n_batches)
        for k in range(n_batches):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


total trainable parameters: 156923


### 8.1 Before training: the random-guessing baseline

Before training, check the untrained model's loss against a theoretical baseline. If a model has learned nothing and is guessing uniformly over the vocabulary, its expected loss is $-\log(1/V) = \log(V)$.

$$
\log(59) = 4.0775
$$

Watch how close the untrained model's actual loss lands to this number, the clearest evidence you'll get that "randomly initialized" really does mean "no better than guessing."


In [15]:
print("theoretical random-guess loss:  log(V) =", round(math.log(vocab_size), 4))

before = estimate_loss()
print("actual untrained model loss:  train =", round(before["train"], 4), " val =", round(before["val"], 4))
print("perplexity (train):", round(math.exp(before["train"]), 4), " -- close to vocab_size =", vocab_size)


theoretical random-guess loss:  log(V) = 4.0775
actual untrained model loss:  train = 4.2098  val = 4.2113
perplexity (train): 67.3438  -- close to vocab_size = 59


Now generate from the untrained model, reusing `sample_decode` from Section 1.2 directly, one character at a time: get the model's logits for the last position, hand them to `sample_decode` exactly as we would have with the toy 5-word vocabulary in Week 3, feed the sampled character back in, repeat.


In [16]:
@torch.no_grad()
def generate(idx, max_new_tokens, temperature=1.0, rng=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        last_logits = logits[0, -1, :].numpy()
        token, probs = sample_decode(last_logits, vocab_list, temperature=temperature, rng=rng)  # Week 3's function
        next_idx = stoi[token]
        idx = torch.cat([idx, torch.tensor([[next_idx]])], dim=1)
    return idx

start = torch.zeros((1, 1), dtype=torch.long)
rng = np.random.default_rng(0)
print("sample generation from the untrained model:")
print(decode(generate(start, max_new_tokens=150, temperature=0.8, rng=rng)[0].tolist()))


sample generation from the untrained model:

bH  mrRhbwi
q
hApVIJ -ieYHzzlcdP-lRHUwuKeEYCMsGf'nlBs!I!No? I-.WEe;wH.dxIxKRczxIiUYlIhkt,gwx
szx:ztoMCpuGWMx ib
h lRu-s!IIyUGBsG-HaSpVHMrUyHWWp'Mu'mKq


Exactly what randomly initialized weights should produce: characters with no relationship to English, spelling, or Shakespeare's style.


### 8.2 Running the training loop

2000 steps is enough for this tiny model to go from the random baseline to something that has clearly picked up spelling patterns, punctuation, and the play's script format, all from character-level next-token prediction with a causal mask, nothing more. Should take roughly a minute or two on CPU.


In [17]:
for step in range(2000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        losses = estimate_loss()
        print(f"step {step:4d}:  train loss {losses['train']:.4f}   val loss {losses['val']:.4f}")

print("training complete.")


step    0:  train loss 3.6526   val loss 3.6178
step  500:  train loss 1.9416   val loss 1.9470
step 1000:  train loss 1.6691   val loss 1.8277
step 1500:  train loss 1.5530   val loss 1.8055
training complete.


### 8.3 After training


In [18]:
after = estimate_loss()
print("train loss:", round(after["train"], 4), " val loss:", round(after["val"], 4))
print("perplexity (train):", round(math.exp(after["train"]), 4))

print("\nsample generation from the trained model:")
print(decode(generate(start, max_new_tokens=300, temperature=0.8, rng=rng)[0].tolist()))


train loss: 1.438  val loss: 1.8439
perplexity (train): 4.2124

sample generation from the trained model:


I theser gilting. Why outcors.

MENENIUS:
But tond the warranes honour abst done noble,
all not was hour, I will not t, was yours?

Messssciance as to Rome, you to be down.

MARCIUS:
How ar
chion tower party hour a may peort form
As trumpeterition of a
cupp your would senceive of their country.

Fi


The loss should have dropped from around 4.2 (random-guess territory) to somewhere near 1.4 to 1.5, and perplexity from roughly 59 down to around 4. The generated text won't be coherent Shakespeare, this model has 3 layers, 64 dimensions, and saw 50,000 characters, but it should show real English word shapes, correct apostrophes and punctuation, and the script's `CHARACTER NAME:` pattern. Nobody told the model what a word is. It found that structure entirely by trying, over and over, to predict the next character through a causally masked attention mechanism it never saw before this course, and getting penalized by cross-entropy when it was wrong.


## 9. Decoding, Revisited

`temperature` here is the same knob from Week 3, Section 2: lower values concentrate probability on the model's top choices (closer to greedy, more repetitive), higher values flatten the distribution (more variety, more mistakes). Try a couple of settings and compare.


In [19]:
for temp in [0.3, 1.2]:
    sample = decode(generate(start, max_new_tokens=150, temperature=temp, rng=rng)[0].tolist())
    print(f"--- temperature={temp} ---")
    print(sample)
    print()


--- temperature=0.3 ---

The could to be the belly.

MENENIUS:
The preserfeces and to the send of his courts in the beard to see in our timperation,
To senswer of his sented m

--- temperature=1.2 ---


There o' the news remenger,' hoad tkime
Lnown.

LERgesld mever?

MARCOMINIUS:
Thence our acctioust talk,
not whont would foort, gooods your mon
wonou



## 10. What Separates This From GPT-2 or GPT-3?

You have now trained a real language model from scratch, using the exact same positional encoding, self-attention, and multi-head attention math from Week 3, plus a genuine causal mask, a genuine transformer block, and a genuine training loop. It's fair to ask what's missing between this and GPT-2. The honest answer is almost nothing conceptual, and almost everything about scale:

- **Data.** GPT-2 trained on roughly 40GB of internet text (billions of tokens). We used 50,000 characters, about the length of a couple of short stories.
- **Parameters.** GPT-2 small has 124 million parameters. Ours has about 157 thousand.
- **Tokenization.** GPT-2 uses BPE (Week 2), giving it a working vocabulary of about 50,000 subword tokens instead of our 59 characters, so each token carries far more meaning.
- **Context length.** GPT-2 conditions on up to 1024 tokens at once. We used 32 characters.
- **Training infrastructure.** GPT-2 trained across many GPUs for days, with learning rate schedules, gradient clipping, and mixed-precision arithmetic we skipped here for simplicity.

None of these are new ideas. Every one is the same forward pass, the same cross-entropy loss, the same backward pass, the same optimizer step you just ran, repeated far more times, on far more data, with a bigger version of the exact architecture from this notebook. Building a real large language model really is a compute and data problem sitting on top of ideas you already understand, not a separate body of knowledge you haven't seen yet.


## 11. Bridge to Week 5

Everything so far has trained a model to do one thing: predict the next token, a generative objective. Starting next week we turn to a different family of tasks: given a full piece of text, predict something *about* it, a category, a sentiment, a topic. That's a discriminative objective, and Week 5 (Text Classification) looks at how the same transformer representations we've been building get repurposed for exactly that.


## 12. Exercise (scaffolded)

Work through these using the `TinyTransformerLM`, `get_batch`, and `estimate_loss` code above.

**Part A: Scale it up or down.**
Change `d_model`, `n_heads`, or `n_layers` and re-run `sum(p.numel() for p in model.parameters())`. Report the new parameter count and, after training for 1000 steps, the new train/val loss. Does a bigger model reach a lower loss in the same number of steps? Does it overfit (train loss much lower than val loss)?

**Part B: Plot the loss curve.**
Modify the training loop to store `losses['train']` and `losses['val']` from every evaluation into two lists, then plot both against training step using matplotlib. Label your axes. At what step does the gap between train and val loss start to widen, if at all?

**Part C: Implement top-k sampling.**
`sample_decode` currently samples from the full distribution. Write a `top_k_sample_decode` variant: before sampling, zero out the probability of every token except the `top_k` highest-probability ones, then renormalize. Compare generations at `top_k=5` against the unrestricted version at the same temperature.

**Part D: Verify the causal mask at scale.**
Pick a real training example from `get_batch("train")`, run a single forward pass, and pull out `weights` from one attention head inside one block (you'll need to temporarily have the block return them). Confirm that for every row $t$, the weights in columns $> t$ are exactly zero. This is the same check from Section 2, now on real data instead of a toy example.

**Part E (optional): a different corpus.**
Swap in a different text file (any public domain text works). Keep everything else fixed. How does the vocabulary size change? How does the generated text's "style" change after training?
